In [1]:
import pandas as pd
import numpy as np

In [2]:
funds = pd.read_csv("../data/raw/01_fund_master.csv")

nav = pd.read_csv("../data/raw/02_nav_history.csv")

transactions = pd.read_csv("../data/raw/08_investor_transactions.csv")

performance = pd.read_csv("../data/raw/07_scheme_performance.csv")

In [3]:
print("Funds:", funds.shape)

print("NAV:", nav.shape)

print("Transactions:", transactions.shape)

print("Performance:", performance.shape)

Funds: (40, 15)
NAV: (46000, 3)
Transactions: (32778, 13)
Performance: (40, 19)


In [4]:
nav["date"] = pd.to_datetime(nav["date"])

print(nav.dtypes)

amfi_code             int64
date         datetime64[ns]
nav                 float64
dtype: object


In [5]:
nav = nav.sort_values(
    ["amfi_code", "date"]
)

nav.head()

,amfi_code,date,nav
5750,100016,2022-01-03,520.4608
5751,100016,2022-01-04,515.0971
5752,100016,2022-01-05,521.7239
5753,100016,2022-01-06,515.7880
5754,100016,2022-01-07,515.1639


In [6]:
duplicates = nav.duplicated().sum()

print("Duplicate Rows:", duplicates)

Duplicate Rows: 0


In [7]:
invalid_nav = (nav["nav"] <= 0).sum()

print("Invalid NAV Values:", invalid_nav)

Invalid NAV Values: 0


In [8]:
print(nav.isnull().sum())

amfi_code    0
date         0
nav          0
dtype: int64


In [9]:
nav["nav"] = nav.groupby("amfi_code")["nav"].ffill()

In [10]:
nav.to_csv(
    "../data/processed/02_nav_history_cleaned.csv",
    index=False
)

print("NAV cleaned and saved successfully")

NAV cleaned and saved successfully


In [11]:
print(transactions["transaction_type"].unique())

['SIP' 'Redemption' 'Lumpsum']


In [12]:
print(transactions["kyc_status"].unique())

['Verified' 'Pending']


In [13]:
print(transactions.isnull().sum())

investor_id           0
transaction_date      0
amfi_code             0
transaction_type      0
amount_inr            0
state                 0
city                  0
city_tier             0
age_group             0
gender                0
annual_income_lakh    0
payment_mode          0
kyc_status            0
dtype: int64


In [14]:
invalid_amounts = (transactions["amount_inr"] <= 0).sum()

print("Invalid Amounts:", invalid_amounts)

Invalid Amounts: 0


In [15]:
transactions["transaction_date"] = pd.to_datetime(
    transactions["transaction_date"]
)

print(transactions["transaction_date"].dtype)

datetime64[ns]


In [26]:
transactions.to_csv(
    "../data/processed/08_investor_transactions_cleaned.csv",
    index=False
)

print("Transactions cleaned and saved")

Transactions cleaned and saved


In [17]:
performance.columns

Index(['amfi_code', 'scheme_name', 'fund_house', 'category', 'plan',
       'return_1yr_pct', 'return_3yr_pct', 'return_5yr_pct',
       'benchmark_3yr_pct', 'alpha', 'beta', 'sharpe_ratio', 'sortino_ratio',
       'std_dev_ann_pct', 'max_drawdown_pct', 'aum_crore', 'expense_ratio_pct',
       'morningstar_rating', 'risk_grade'],
      dtype='object')

In [18]:
print(performance.isnull().sum())

amfi_code             0
scheme_name           0
fund_house            0
category              0
plan                  0
return_1yr_pct        0
return_3yr_pct        0
return_5yr_pct        0
benchmark_3yr_pct     0
alpha                 0
beta                  0
sharpe_ratio          0
sortino_ratio         0
std_dev_ann_pct       0
max_drawdown_pct      0
aum_crore             0
expense_ratio_pct     0
morningstar_rating    0
risk_grade            0
dtype: int64


In [24]:
numeric_cols = [
    "return_1yr_pct",
    "return_3yr_pct",
    "return_5yr_pct",
    "benchmark_3yr_pct",
    "alpha",
    "beta",
    "sharpe_ratio",
    "sortino_ratio",
    "std_dev_ann_pct",
    "max_drawdown_pct",
    "aum_crore",
    "expense_ratio_pct"
]

for col in numeric_cols:
    performance[col] = pd.to_numeric(
        performance[col],
        errors="coerce"
    )

print(performance[numeric_cols].dtypes)

return_1yr_pct       float64
return_3yr_pct       float64
return_5yr_pct       float64
benchmark_3yr_pct    float64
alpha                float64
beta                 float64
sharpe_ratio         float64
sortino_ratio        float64
std_dev_ann_pct      float64
max_drawdown_pct     float64
aum_crore              int64
expense_ratio_pct    float64
dtype: object


In [19]:
invalid_expense = performance[
    (performance["expense_ratio_pct"] < 0.1) |
    (performance["expense_ratio_pct"] > 2.5)
]

print("Invalid Expense Ratios:", len(invalid_expense))

Invalid Expense Ratios: 0


In [20]:
anomalies = performance[
    (performance["return_1yr_pct"] > 100) |
    (performance["return_1yr_pct"] < -50)
]

print("Return Anomalies:", len(anomalies))

Return Anomalies: 0


In [25]:
performance.to_csv(
    "../data/processed/07_scheme_performance_cleaned.csv",
    index=False
)

print("Performance dataset cleaned and saved")

Performance dataset cleaned and saved
